In [ ]:
!pip install torch

In [ ]:
!pip uninstall pyarrow -y

In [1]:
!pip install elasticsearch==8.11.0

DEPRECATION: celery 5.0.5 has a non-standard dependency specifier pytz>dev. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of celery or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063



  Using cached elasticsearch-8.11.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached elastic_transport-8.17.1-py3-none-any.whl.metadata (3.8 kB)
Using cached elasticsearch-8.11.0-py3-none-any.whl (412 kB)
Using cached elastic_transport-8.17.1-py3-none-any.whl (64 kB)


In [2]:
from elasticsearch import Elasticsearch

# Tạo client
es = Elasticsearch(['http://34.171.201.34:9200'])

In [3]:
if es.ping():
    print("✅ Kết nối thành công!")
else:
    print("❌ Không thể kết nối đến Elasticsearch.")

✅ Kết nối thành công!


In [4]:
valid_districts = [
        "Ba Đình", "Bắc Từ Liêm", "Cầu Giấy", "Đống Đa", "Hà Đông", "Hai Bà Trưng",
        "Hoàn Kiếm", "Hoàng Mai", "Long Biên", "Nam Từ Liêm", "Tây Hồ", "Thanh Xuân",
        "Ba Vì", "Chương Mỹ", "Đan Phượng", "Đông Anh", "Gia Lâm", "Hoài Đức",
        "Mê Linh", "Mỹ Đức", "Phú Xuyên", "Phúc Thọ", "Quốc Oai", "Sóc Sơn",
        "Thạch Thất", "Thanh Oai", "Thanh Trì", "Thường Tín", "Ứng Hòa", "Sơn Tây"
    ]

In [5]:
def get_price_by_district(estate_type="nhapho"):
    """
    Lấy dữ liệu giá trung bình theo quận/huyện
    Args:
        estate_type (str): Loại nhà ('nhapho', 'nharieng', 'chungcu' hoặc 'bietthu')
    Returns:
        DataFrame: DataFrame chứa thông tin quận/huyện và giá trung bình
    """
    index_mapping = {
        "nhapho": "nhapho_index",
        "nharieng": "nharieng_index",
        "chungcu": "chungcu_index",
        "bietthu": "bietthu_index"
    }
    
    index_name = index_mapping.get(estate_type, "nhapho_index")

    query = {
        "size": 0,
        "query": {
            "terms": {
                "address.district.keyword": valid_districts
            }
        },
        "aggs": {
            "group_by_district": {
                "terms": {
                    "field": "address.district.keyword",
                    "size": len(valid_districts)
                },
                "aggs": {
                    "avg_price": {
                        "avg": {
                            "field": "price"
                        }
                    }
                }
            }
        }
    }
    
    response = es.search(index=index_name, body=query)
    
    districts = valid_districts
    avg_prices = []

    avg_price_by_district = {district: 0 for district in valid_districts}

    # Cập nhật giá trị trung bình từ kết quả truy vấn
    for bucket in response['aggregations']['group_by_district']['buckets']:
        district = bucket['key']
        avg_price = bucket['avg_price']['value']
        if avg_price is not None:
            avg_price_by_district[district] = avg_price

    for district in districts:
        avg_prices.append(avg_price_by_district[district])

    return districts, avg_prices

In [6]:
d, a = get_price_by_district()

In [8]:
a

[70261324616.50415,
 41502567897.51984,
 60989585162.5581,
 44911508098.25214,
 24935390912.6287,
 66238494817.70909,
 131203379052.49327,
 24605620644.741745,
 30214446648.040886,
 45996859570.833786,
 78046223117.94911,
 37841743305.304825,
 5440000102.4,
 8600000076.8,
 13655555584.0,
 17006883666.753489,
 18228751168.31841,
 12824433595.469027,
 7994814862.222222,
 8425000192.0,
 5500000256.0,
 8100000256.0,
 5634285714.285714,
 14428571446.857143,
 11000000000.0,
 5450000128.0,
 20377619641.47451,
 14580000000.0,
 9200000000.0,
 10348346777.290323]